# Custom output and blueprints


## Introduction

In addition to the `Standard Output` Amazon Bedrock Data Automation (BDA) offers the `Custom Output` feature which lets you define the target structure for information which you want to extract or generate from documents or images. This capability is particularly useful when working with specialized, or complex semi structured documents.

You can configure custom output in BDA by using `Blueprints`. `Blueprints` are essentially a lists of instructions and types that guide the extraction or generation of information based on your documents. This feature works in conjunction with BDA projects, enabling the processing of up to 40 page document input and one image input. 

Custom outputs provide users with greater control and flexibility to derive structured information from their documents towards particular use cases or flows.

### Blueprints

You can use blueprints to configure file processing business logic in Amazon Bedrock Data Automation (BDA). Each blueprint consists of a list of field names to extract, the desired data format for each field (e.g., string, number, boolean), and natural language context for data normalization and validation rules. 

The main fields for creating blueprint are:

```python
response = client.create_blueprint(
    blueprintName='string',
    type='DOCUMENT'|'IMAGE',
    blueprintStage='DEVELOPMENT'|'LIVE',
    schema='string', # Schema of the blueprint (fields, groups, tables, etc.)
)
```

BDA has ready-to-use blueprints (`Catalog Blueprints`) for a number of commonly used document types such as W2, Paystub or a Receipt. Catalog blueprints are a great way to start if the document you want to extract from matches the blueprint. To extract from documents that are not matched by blueprints in the catalog you can create your own blueprints. When creating the blueprint using the AWS Console, you have the option to let BDA generate blueprint after providing a sample document and an optional prompt. You can also create the blueprint by adding individual fields or by using a JSON editor to define the JSON for the blueprint.

In this notebook, we will explore custom output using blueprints and data automation projects.

### Prerequisites

In [ ]:
pip install "boto3>=1.35.76" PyPDF2 pypdfium2 itables --upgrade -qq

### Setup

Before we get to the part where we invoke BDA with our sample artifacts, let's setup some parameters and configuration that will be used throughout this notebook

In [ ]:
import boto3
import json
import pprint
from IPython.display import JSON, display, HTML, Markdown, IFrame
import IPython.display as display
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
from itables import show
import json


default_execution_role = get_execution_role()
session = sagemaker.Session()
default_bucket = session.default_bucket()

# Initialize Bedrock Data Automation client
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

bda_s3_input_location = f's3://{default_bucket}/bda/input'
bda_s3_output_location = f's3://{default_bucket}/bda/output'
print(f'Using default execution role {default_execution_role}')

In [ ]:
%load_ext autoreload
%autoreload 1

## Custom output using catalog blueprint

Now that we have our sample document available in S3, let's start with using the blueprints. Bedrock offers sample blueprints for some common document types such as W2, pay stub or bank statement. To begin with, we'll use a sample bank statement 

### View Sample Document

In [ ]:
document_local = "data/documents/BankStatement.jpg"
IFrame(document_local, width=640, height=480)

### Upload sample document to S3

Lets use a banking statement to explore the blueprint feature of BDA. 

In [ ]:
from utils.helper_functions import wait_for_job_to_complete, read_s3_object, download_document, get_bucket_and_key, wait_for_completion, display_html

input_bucket, input_prefix = get_bucket_and_key(bda_s3_input_location)
local_file_name = 'data/documents/BankStatement.jpg'
s3_file_name = 'BankStatement.jpg'
input_file_s3 = f's3://{input_bucket}/{input_prefix}/{s3_file_name}'
s3_response = s3_client.upload_file(local_file_name, input_bucket, f"{input_prefix}/{s3_file_name}")

### List blueprints in the catalog
Let's view the blueprints that BDA offers in the catalog of sample blueprints

In [ ]:
list_blueprints_response = bda_client.list_blueprints(resourceOwner='SERVICE')
df = pd.DataFrame(list_blueprints_response['blueprints'])[['blueprintName','blueprintArn']]
show(df)

### Invoke Blueprint Recommendation
With our sample ready, we can have BDA recommend a blueprint for our sample document from the sample set.

In [ ]:
from utils.helper_functions import invoke_blueprint_recommendation_async, get_blueprint_recommendation
import sys
sys.path.append('..')
%aimport utils.helper_functions 

inputConfiguration = {
    "inputDataConfiguration":{
        "s3Uri":input_file_s3
    }
}
print(f"Invoking blueprint recommendation for {input_file_s3}")
response = invoke_blueprint_recommendation_async(bda_client=bda_client, payload=json.dumps(inputConfiguration))

job_id = response['jobId']

### Wait for blueprint recommendation results

In [ ]:
status_response = wait_for_completion(
            client=None,
            get_status_function=get_blueprint_recommendation,
            status_kwargs={
                'bda_client': bda_client,
                'job_id': job_id,
            },
            completion_states=['Completed'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)

### Identify Blueprint
BDA outputs a blueprint recommendation along with a prompt recommendation that is useful to create a custom blueprint, if needed.

For this example, we will fetch the blueprint that was recommended by BDA.

In [ ]:
blueprint_recommendation = next((result for result in status_response['results'] if result['type'] == 'BLUEPRINT_RECOMMENDATION'),None)

In [ ]:
JSON(blueprint_recommendation['blueprintRecommendation'], root='blueprintRecommendation', expanded=True)

### Blueprint Schema

Now that we have identified the matching Blueprint, we can view the blueprint schema. The blueprint schema describes the data structure that contains fields, which in turn contain the information extracted by BDA custom output. There are two types of fields—explicit and generative—located in the extraction table. Explicit extractions are used for clearly stated information that can be seen in the document. Generative extractions are used for information that need to be transformed from how they appear in the document

In [ ]:
JSON(json.loads(blueprint_recommendation['blueprintRecommendation']['schema']), root='Schema', expanded=False)

In [ ]:
blueprint_arn = blueprint_recommendation['blueprintRecommendation']['matchedBlueprint']['blueprintArn']

### Invoke Data Automation Async
Now that we have identified a blueprint, we can proceed to invoke data automation. Note that in addition to the input and output configuration we also provide the blueprint id when calling the `invoke_data_automation_async` operation.

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': f'{bda_s3_input_location}/{s3_file_name}'
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    blueprints=[
        {
            'blueprintArn': blueprint_arn
        }
    ]
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

### Get Data Automation Status

We can check the status and monitor the progress of the Invocation job using the `GetDataAutomationStatus`. This API takes the invocation arn we retrieved from the response to the `InvokeDataAutomationAsync` operation above.

The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully, along with the S3 location of the results. If the job encounters and error the final status is either `ServiceError` or `ClientError` with error details

In [ ]:
status_response = wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

### Get job metadata for custom output

In [ ]:
job_metadata = json.loads(read_s3_object(job_metadata_s3_location))
JSON(job_metadata, root='job_metadata', expanded=True)

### Explore Custom output of Blueprint 
We can now explore the custom output received from processing documents using the blueprint we used for the Data Automation job.

Note,that Standard output is always produced.

Let's break down the main sections of this JSON output from Bedrock Data Automation:

In [ ]:
asset_id = 0
custom_output_path = next(item["segment_metadata"][0]["custom_output_path"] 
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)
custom_output = json.loads(read_s3_object(custom_output_path))

In [ ]:
JSON(custom_output)

#### _matched_blueprint_

- Contains information about the template used for analysis
- Includes the ARN, name ("Bank-Statement"), and confidence score for the match.

The confidence score is the degree of certainty with which BDA has matched the provided document to a blueprint.

Note: Since we passed in the blueprint arn in the `invoke_data_automation_async` BDA uses the blueprint with that Arn and hence the confidence score is 1. Later in this notebook, when using projects, we will see an example where BDA determines a blueprint for a document from a configured set of blueprints.

In [ ]:
JSON(custom_output['matched_blueprint'], root='matched_blueprint')

#### _document class_

In [ ]:
JSON(custom_output['document_class'], root='document_class')

#### _inference_results_

Inference results section contain the data BDA extracted from the document using the blueprint provided.

In [ ]:
JSON(custom_output['inference_result'], root='inference_result')

#### _explainability_info_
Explanabilitx results section contain the support information to each extracted item, like confidence score and bounding box.

In [ ]:
JSON(custom_output['explainability_info'], root='explainability_info')

### Side-by-side visualization of the original file and extracted information

In [ ]:
import pypdfium2 as pdfium
from utils.helper_functions import get_s3_to_dict, display_image_jsons
import ipywidgets as widgets
from PIL import Image

if local_file_name.endswith(".pdf"):
    doc = pdfium.PdfDocument(local_file_name)
    pages_pil = [page.render(scale=1.53).to_pil() for page in doc]
else:
    pages_pil = [Image.open(local_file_name)]

# get the job_metadata
job_json_obj = json.loads(read_s3_object(job_metadata_s3_location))
results_meta = job_json_obj["output_metadata"][0]["segment_metadata"]

# put the results together and show with first page side by side
results_all = []
for result in results_meta:
    standard_output_obj = get_s3_to_dict(result["standard_output_path"])
    custom_output_obj = get_s3_to_dict(result["custom_output_path"])
    pages = custom_output_obj["split_document"]["page_indices"]
    w = display_image_jsons(pages_pil[pages[0]], [custom_output_obj['matched_blueprint'],custom_output_obj['inference_result']],["Matched Blueprint", "Inference Result"])
    results_all.append(w)    

widgets.VBox(results_all)


---

## Custom output using custom blueprint

For documents and images that aren't in the catalog, you can create custom blueprints. In the following example, we will extract data from a sample medical claim form along using a blueprint that we create.

### View Sample Document

In [ ]:
IFrame("data/documents/sample1_cms-1500-P.pdf", width=900, height=800)

### Upload sample document to S3

Lets use a CMS 1500 Medical claim to explore the blueprint feature of BDA. 

In [ ]:
input_bucket, input_prefix = get_bucket_and_key(bda_s3_input_location)
local_file_name = 'data/documents/sample1_cms-1500-P.pdf'
s3_file_name = 'sample1_cms-1500.pdf'
input_file_s3 = f's3://{input_bucket}/{input_prefix}/{s3_file_name}'
s3_response = s3_client.upload_file(local_file_name, input_bucket,
                                    f'{input_prefix}/{s3_file_name}')

### Invoke Blueprint Recommendation

Before we start creating our own blueprint, let's explore the Blueprint recommendation with our sample document

In [ ]:
from utils.helper_functions import invoke_blueprint_recommendation_async, get_blueprint_recommendation
import json
inputConfiguration = {
    "inputDataConfiguration":{
        "s3Uri":f'{bda_s3_input_location}/{s3_file_name}'
    }
}
response = invoke_blueprint_recommendation_async(bda_client=bda_client,                                      
                                      payload=json.dumps(inputConfiguration))

job_id = response['jobId']

### Wait for blueprint recommendation results

In [ ]:
status_response = wait_for_completion(
            client=None,
            get_status_function=get_blueprint_recommendation,
            status_kwargs={
                'bda_client': bda_client,
                'job_id': job_id,                
            },
            completion_states=['Completed'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)

In [ ]:
blueprint_recommendation = next((result for result in status_response['results'] if result['type'] == 'BLUEPRINT_RECOMMENDATION'),None)
JSON(blueprint_recommendation['blueprintRecommendation'],root='blueprintRecommendation',expanded=False)

Note that BDA identified the type of the sample file provided as `DOCUMENT` and the document class as 'Health Insurance Claim Form' (or similar). However, the `matchedBlueprint` section is missing, indicating that BDA did not find an existing blueprint in the BDA catalog of premade blueprints.

Now, Let's start by creating our first blueprint.

### Define Blueprint properties
To create a blueprint you start with defining a blueprint name, description, the blueprint type (`DOCUMENT` or `IMAGE`), the blueprint stage (`LIVE` or `DEVELOPMENT`) along with blueprint schema in JSON schema format.

 You can create a blueprint using an API providing a name, type, stage and a schema in JSON format.

In [ ]:
# create blueprint using Boto3
blueprint_name = 'medical-claim-form-cms1500'
blueprint_description = 'Blueprint for CMS 1500 Claim Form'	
blueprint_type = 'DOCUMENT'
blueprint_stage = 'LIVE'

with open('data/blueprints/blueprint_schema.json') as f:
    blueprint_schema = json.load(f)
JSON(blueprint_schema)

### Create (or Update) Blueprint

We will use the `create_blueprint` operation (or `update_blueprint` to update an existing blueprint) in the  `boto3` API to create/update the blueprint. You could also create/update blueprints using the AWS console. Each blueprint that you create is an AWS resource with its own blueprint ID and ARN. 

In [ ]:
list_blueprints_response = bda_client.list_blueprints(
    blueprintStageFilter='ALL'
)
blueprint = next((blueprint for blueprint in
                  list_blueprints_response['blueprints']
                  if 'blueprintName' in blueprint and
                  blueprint['blueprintName'] == blueprint_name), None)


print(f'Found existing blueprint with name={blueprint_name} and arn={blueprint_arn}, updating Stage and Schema')

if not blueprint:
    response = bda_client.create_blueprint(
        blueprintName=blueprint_name,
        type=blueprint_type,
        blueprintStage=blueprint_stage,
        schema=json.dumps(blueprint_schema)
    )
else:
    response = bda_client.update_blueprint(
        blueprintArn=blueprint['blueprintArn'],
        blueprintStage=blueprint_stage,
        schema=json.dumps(blueprint_schema)
    )

blueprint_arn = response['blueprint']['blueprintArn']

### Invoke Data Automation Async
Now that our blueprint has been setup, we can proceed to invoke data automation. Note that in addition to the input and output configuration we also provide the blueprint Arn when calling the `invoke_data_automation_async` operation.

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': f'{bda_s3_input_location}/{s3_file_name}'
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    blueprints=[
        {
            'blueprintArn': blueprint_arn
        }
    ]
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}') 

### Get Data Automation Status

We can check the status and monitor the progress of the Invocation job using the `GetDataAutomationStatus`. This API takes the invocation arn we retrieved from the response to the `InvokeDataAutomationAsync` operation above.

The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully, along with the S3 location of the results. If the job encounters and error the final status is either `ServiceError` or `ClientError` with error details

In [ ]:
status_response = wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

In [ ]:
job_metadata = json.loads(read_s3_object(job_metadata_s3_location))
JSON(job_metadata, root='job_metadata', expanded=True)

### Explore the Custom output with custom blueprint

In [ ]:
asset_id=0
custom_output_path = next(item["segment_metadata"][0]["custom_output_path"] 
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)
custom_output = json.loads(read_s3_object(custom_output_path))
JSON(custom_output)

The structure of the custom output would be the same as that of the output produced when using a catalog blueprint. However, the `inference_result` now contain data that map to the blueprint schema we provided to BDA with the `InvokeDataAutomationAsync` operation.

### Side-by-side visualization of the original file and extracted information

In [ ]:
import pypdfium2 as pdfium
from utils.helper_functions import get_s3_to_dict, display_image_jsons
import ipywidgets as widgets
from PIL import Image


if local_file_name.endswith(".pdf"):
    doc = pdfium.PdfDocument(local_file_name)
    pages_pil = [page.render(scale=1.53).to_pil() for page in doc]
else:
    pages_pil = [Image.open(local_file_name)]

# get the job_metadata
job_json_obj = json.loads(read_s3_object(job_metadata_s3_location))
results_meta = job_json_obj["output_metadata"][0]["segment_metadata"]

# put the results together and show with first page side by side
results_all = []
for result in results_meta:
    standard_output_obj = get_s3_to_dict(result["standard_output_path"])
    custom_output_obj = get_s3_to_dict(result["custom_output_path"])
    pages = custom_output_obj["split_document"]["page_indices"]
    w = display_image_jsons(pages_pil[pages[0]], [custom_output_obj['matched_blueprint'],custom_output_obj['inference_result']],["Matched Blueprint", "Inference Result"])
    results_all.append(w)    

widgets.VBox(results_all)
